# 🌦️ Cambodia Weather Forecast Analysis
### Fundamentals of Data Science Project
Cambodia Academy of Digital Technology (Group 4)

This notebook analyzes weather trends in Cambodia using government open data to explore climate patterns and forecasting insights.

In [ ]:
# import Library
import sys
from pathlib import Path
from datetime import datetime, timedelta
import time

import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'notebooks' / '_shared' / 'notebook_utils.py').exists():
    project_root = project_root.parent

shared_dir = project_root / 'notebooks' / '_shared'
if str(shared_dir) not in sys.path:
    sys.path.append(str(shared_dir))

from notebook_utils import add_time_features, eda_summary_table, load_weather_csv, missing_values_report

# plot style for cleaner visuals
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Fetch Weather Data
Collect real-time weather data from Cambodia government API for analysis of temperature, humidity, and climate trends.

In [ ]:
# Provinces and coordinates
locations = [
    ("Phnom Penh", 11.55, 104.91),
    ("Siem Reap", 13.36, 103.85),
    ("Battambang", 13.10, 103.20),
    ("Sihanoukville", 10.62, 103.52),
    ("Mondulkiri", 12.45, 107.20),
]

# Request configuration
url = "https://archive-api.open-meteo.com/v1/archive"
start_date = "2015-01-01"
end_date = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d") # safe API

all_data = [] 

for province, lat, lon in locations:
    print(f"Fetching data for {province}...")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "wind_speed_10m_max",
        ],
        "timezone": "Asia/Bangkok",
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()

        province_df = pd.DataFrame(payload["daily"])
        province_df["province"] = province
        province_df["lat"] = lat
        province_df["lon"] = lon
        all_data.append(province_df)

        time.sleep(0.8)  # Respectful delay between requests

    except Exception as err:
        print(f"Skipped {province} due to error: {err}")

if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.rename(
        columns={
            "time": "date",
            "temperature_2m_max": "temp_max",
            "temperature_2m_min": "temp_min",
            "precipitation_sum": "rain",
            "wind_speed_10m_max": "wind_speed",
        }
    )

    output_path = "../data/cambodia_weather.csv"
    final_df.to_csv(output_path, index=False)
    print(f"Saved {len(final_df):,} rows to {output_path}")
else:
    final_df = pd.DataFrame()
    print("No data retrieved from API.")

### 2. Exploratory Data Analysis (Step-by-Step)
This section is organized for clear understanding of the dataset before deeper analysis:
1. Load the dataset for EDA.
2. Check total rows and columns.
3. Inspect column names and data types.
4. Validate missing values and date coverage.
5. Continue with statistics and visualization.

In [ ]:
# Load dataset
try:
    df = load_weather_csv("../data/cambodia_weather.csv")
    print("Dataset loaded from ../data/cambodia_weather.csv")
except FileNotFoundError:
    df = final_df.copy()
    if not df.empty:
        df.columns = [col.strip() for col in df.columns]
    print("CSV not found, using in-memory final_df instead")

# rows and columns
rows, cols = df.shape
print(f"Total rows: {rows:,}")
print(f"Total columns: {cols}")

df.head()

In [ ]:
# Display dataframe shape and summary info
print("=" * 60)
print("DATAFRAME SUMMARY")
print("=" * 60)
print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nDataFrame Info:")
df.info()
print(f"\nData Types:\n{df.dtypes}")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"\nUnique Provinces: {df['province'].nunique()} ({', '.join(df['province'].unique())})")
print(f"\nFirst 5 rows:")
display(df.head())
print(f"\nLast 5 rows:")
display(df.tail())

### Step 3. Inspect Columns and Data Types
Now we check what each column represents and confirm data types before cleaning.

In [ ]:
# Column list and data types
print("Columns in dataset:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

# Convert and prepare key fields for analysis
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "temp_max", "temp_min", "rain", "wind_speed", "province"])
df = df.drop_duplicates().sort_values(["province", "date"]).reset_index(drop=True)
df = add_time_features(df, date_col="date")
df["temp_avg"] = (df["temp_max"] + df["temp_min"]) / 2

print(f"\nShape after preparation: {df.shape}")

In [ ]:
# Statistical summary of numeric columns
print("=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
print(f"\nDescriptive Statistics:\n")
display(df[["temp_max", "temp_min", "rain", "wind_speed"]].describe().round(2))

In [ ]:
# Data quality checks - missing values and duplicates
print("=" * 60)
print("DATA QUALITY METRICS")
print("=" * 60)
print(f"\nMissing Values:\n")
quality_df = missing_values_report(df)
display(quality_df[quality_df["Missing Count"] > 0] if quality_df["Missing Count"].sum() > 0 else "No missing values!")
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
print(f"\nData is clean and ready for analysis!")

### Step 4. Data Quality Check
After type conversion and cleaning, verify missing values and date coverage.

In [ ]:
# Missing-value report
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False))

# Date coverage check
print("\nDate coverage:")
print(f"Start date: {df['date'].min().date()}")
print(f"End date:   {df['date'].max().date()}")

# Province-level descriptive statistics
summary_stats = (
    df.groupby("province")[["temp_max", "temp_min", "temp_avg", "rain", "wind_speed"]]
    .agg(["mean", "median", "std"])
    .round(2)
)
summary_stats

### Step 4. Visual Analysis
We use visual EDA to understand variability and trends:
- Temperature spread across provinces.
- Monthly rainfall pattern.
- Yearly temperature trend by province.

In [ ]:
# Temperature distribution by province
plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x="province", y="temp_avg", palette="Set2")
plt.title("Average Daily Temperature by Province")
plt.xlabel("Province")
plt.ylabel("Temperature (deg C)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Step 5. Temporal Trends
We now evaluate weather behavior over time to reveal seasonal and long-term signals useful for forecasting.

#### 5.1 Monthly Rainfall Trend (All Provinces Combined)

In [ ]:
# Monthly rainfall trend
monthly_rain = df.set_index("date").resample("ME")["rain"].mean()

plt.figure(figsize=(12, 5))
monthly_rain.plot(color="#2b8cbe", linewidth=1.8)
plt.title("Monthly Average Rainfall Across Selected Provinces")
plt.xlabel("Date")
plt.ylabel("Rainfall (mm)")
plt.tight_layout()
plt.show()

# Yearly average temperature trend by province
yearly_temp = (
    df.groupby(["year", "province"])["temp_avg"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=yearly_temp, x="year", y="temp_avg", hue="province", marker="o")
plt.title("Yearly Average Temperature Trend by Province")
plt.xlabel("Year")
plt.ylabel("Temperature (deg C)")
plt.tight_layout()
plt.show()

In [ ]:
# latest province snapshot for quick comparison
latest_snapshot = (
    df.sort_values("date")
    .groupby("province")
    .tail(1)
    [["province", "date", "temp_max", "temp_min", "temp_avg", "rain", "wind_speed"]]
    .reset_index(drop=True)
)
latest_snapshot

### 5.2 Feature Relationship Visualization
To better understand forecasting signals, we visualize numeric feature correlations and monthly temperature seasonality.

In [ ]:
# Correlation heatmap for key numeric features
corr_cols = ['temp_max', 'temp_min', 'temp_avg', 'rain', 'wind_speed', 'lat', 'lon', 'month', 'dayofweek']
corr_matrix = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of Weather and Time Features')
plt.tight_layout()
plt.show()

# Monthly seasonality of average temperature
plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x='month', y='temp_avg', color='#74a9cf')
plt.title('Monthly Temperature Seasonality (temp_avg)')
plt.xlabel('Month')
plt.ylabel('Temperature (deg C)')
plt.tight_layout()
plt.show()

### 6. EDA Summary

This section generates a concise, report-ready summary of:
- EDA key facts (rows, date range, provinces, data quality)
- Climate statistics (temperature, rainfall, wind)
- Data completeness and readiness for downstream modeling


In [ ]:
# Report-ready EDA summary
if "df" not in globals():
    df = load_weather_csv("../data/cambodia_weather.csv")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", "temp_max", "temp_min", "rain", "wind_speed", "province"])
    df = df.drop_duplicates().sort_values(["province", "date"]).reset_index(drop=True)
    df = add_time_features(df, date_col="date")
    df["temp_avg"] = (df["temp_max"] + df["temp_min"]) / 2

eda_summary_df = eda_summary_table(df)

print("EDA SUMMARY")
display(eda_summary_df)